In [1]:
import openai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import re
import time
from openai import OpenAI
import json
import pickle
import pandas as pd
import os

sys.path += ['../src/']

#from model import *
import model as mod
import model
from utils import *

In [2]:
path_exp = '../results/experiments/'
openai.__version__

'1.50.2'

In [3]:
# set simulation parameters
seed = 0
np.random.seed(seed)
noise_mean = 0
noise_sd = 1/4
temperature = 1
memory = 3
instruction_type = 'original_30-50w'
n_agents = 6
n_steps = 50
#gpt_model="gpt-4"
gpt_model="gpt-3.5-turbo-0125"

In [4]:
expmnt_num = 0
feedback='bub'
expmnt_num = check_existing_file(path_exp, expmnt_num, feedback, instruction_type, temperature, memory, n_steps, n_agents, gpt_model)

File expmnt_results_bub_original_30-50w_num_0_temp_1_memory_3_ntime_50_nagents_6_model_gpt-3-5-turbo-0125.csv does not exist. Safe to proceed.


In [5]:
noise_mean=3
noise_sd=1/4

In [6]:
mod.f_bubbles([50, 47.25, 56.75])

np.float64(52.16604420935738)

In [7]:
mod.f_bubbles([37.75, 50.,   46.2])

np.float64(45.4762279067541)

In [8]:
expmnt_num=0
p_array_op, pe_agents_time_op, rewards_agents_time_op, messages_list_op  = mod.run_experiment(seed,\
                expmnt_num, noise_mean, noise_sd, temperature, memory=memory, gpt_model=gpt_model, n_steps=n_steps, n_agents=n_agents,
                feedback=feedback, instruction_type=instruction_type, random_start=True, prints_on=True,\
                save_res_bool=True, path_exp=path_exp)

 
 Time: 0, agent: 0 

{'role': 'user', 'content': 'This is the first time step, give an initial random price prediction for the first two periods.  It is very likely that the stock price will be between 0 and 100 in the first two periods.'}
{'role': 'assistant', 'content': '{\n\t"reasoning": "For the initial prediction, I will randomly select a value between 0 and 100 for each period.",\n\t"predictedValue1": 45.75,\n\t"predictedValue2": 68.23\n}'}
 
 Time: 0, agent: 1 

{'role': 'user', 'content': 'This is the first time step, give an initial random price prediction for the first two periods.  It is very likely that the stock price will be between 0 and 100 in the first two periods.'}
{'role': 'assistant', 'content': '{\n\t"reasoning": "As this is the initial prediction without any historical data or context, a random prediction is made within the likely range of 0 and 100 for the first two periods.",\n\t"predictedValue1": 45.67,\n\t"predictedValue2": 57.89\n}'}
 
 Time: 0, agent: 2 


OSError: Cannot save file into a non-existent directory: '..\results\experiments'

In [ ]:
expmnt_nums = [1, 2, 3]
for expmnt_num in expmnt_nums:
    p_array_op, pe_agents_time_op, rewards_agents_time_op, messages_list_op  = mod.run_experiment(seed,\
                expmnt_num, noise_mean, noise_sd, temperature, memory=memory, gpt_model=gpt_model, n_steps=n_steps, n_agents=n_agents,
                feedback=feedback, instruction_type=instruction_type, random_start=True, prints_on=True,\
                save_res_bool=True, path_exp=path_exp)

In [ ]:

def plots_across_runs_and_feedback(feedback_values, expmnt_nums, gpt_model, memory, temperature):
    n = 2  # number of rows (positive and negative feedback)
    m = 3  # number of columns (three runs)

    fig = plt.figure(figsize=(15, 10))
    gs = GridSpec(n, m, figure=fig)
    
    max_price_pos = 0
    min_price_pos = 100
    max_price_neg = 0
    min_price_neg = 100
    
    for feedback in feedback_values:
        for expmnt_num in expmnt_nums:
            try:
                df = pd.read_csv(f"../results/experiments/expmnt_results_{feedback}_original_30-50wrandstart_num_{expmnt_num}_temp_{str(temperature).replace('.', '-')}_memory_{memory}_ntime_50_nagents_6.csv")
            except:
                try:
                    df = pd.read_csv(f"../results/experiments/expmnt_results_{feedback}_original_30-50wrandstart_num_{expmnt_num}_temp_{str(temperature).replace('.', '-')}_memory_{memory}_ntime_50_nagents_6_model_{str(gpt_model).replace('.', '-')}.csv")
                except:
                    df = pd.read_csv(f"../results/experiments/expmnt_results_{feedback}_original_30-50wrandstart_num_{expmnt_num}_temp_{str(1)}_memory_{memory}_ntime_50_nagents_6_model_{str(gpt_model).replace('.', '-')}.csv")
            if feedback == "pos":
                max_price_pos = max(max_price_pos, df["predicted_price"].max(), df["actual_price"].max())
                min_price_pos = min(min_price_pos, df["predicted_price"].min(), df["actual_price"].min())
    
            if feedback == "neg":
                max_price_neg = max(max_price_neg, df["predicted_price"].max(), df["actual_price"].max())
                min_price_neg = min(min_price_neg, df["predicted_price"].min(), df["actual_price"].min())
    
    for i, feedback in enumerate(feedback_values):
        for j, expmnt_num in enumerate(expmnt_nums):
            ax = fig.add_subplot(gs[i, j])
            try:
                df = pd.read_csv(f"../results/experiments/expmnt_results_{feedback}_original_30-50wrandstart_num_{expmnt_num}_temp_{str(temperature).replace('.', '-')}_memory_{memory}_ntime_50_nagents_6.csv")
            
            except:
                try:
                    df = pd.read_csv(f"../results/experiments/expmnt_results_{feedback}_original_30-50wrandstart_num_{expmnt_num}_temp_{str(temperature).replace('.', '-')}_memory_{memory}_ntime_50_nagents_6_model_{str(gpt_model).replace('.', '-')}.csv")
                except:
                    
                    df = pd.read_csv(f"../results/experiments/expmnt_results_{feedback}_original_30-50wrandstart_num_{expmnt_num}_temp_{str(1)}_memory_{memory}_ntime_50_nagents_6_model_{str(gpt_model).replace('.', '-')}.csv")
            # Plot actual and predicted prices over time_step
            ax.plot(df['time_step'], df['actual_price'], label='Actual Price', color="black", linewidth=3)
            for agent_id, group in df.groupby('agent_id'):
                ax.plot(group['time_step'], group['predicted_price'], label=f'Predicted Price (Agent {agent_id})')
            
            if j == 0:
                ax.set_ylabel('Price')
            if i == 1:
                ax.set_xlabel('Time Step')
            ax.set_title(f"Run: {j+1}")
            ax.axhline(60, linestyle="dashed", color="black", alpha=0.5)
            # if j == 2:
            #     ax.legend(bbox_to_anchor=[1.05, 1])
            if i == 0:
                ax.set_ylim(0, 100)
            if i == 1:
                ax.set_ylim(0, 100)
    
    plt.suptitle(f"Memory: {memory}; Temperature: {temperature}", fontsize=20)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    filename = f"../results/plots/{gpt_model}_memory_{memory}_temperature_{temperature}_alternative_layout_randstart.png"
    plt.savefig(filename, bbox_inches='tight')
    plt.show()


In [8]:
#op =original positive
gpt_model = "gpt-4-1106-preview"
gpt_model = "gpt-3.5-turbo"
# gpt_model="gpt-4"
expmnt_nums = [1, 2, 3]
for expmnt_num in expmnt_nums:
    p_array_op, pe_agents_time_op, rewards_agents_time_op, messages_list_op  = mod.run_experiment(seed,\
                expmnt_num, noise_mean, noise_sd, temperature, memory=memory, gpt_model=gpt_model, n_steps=n_steps, n_agents=n_agents,
                feedback=feedback, instruction_type=instruction_type, prints_on=True,\
                save_res_bool=True, path_exp=path_exp)

 
 Time: 0, agent: 0 

{'role': 'user', 'content': 'This is the first time step, give an initial price prediction for the first two periods.  It is very likely that the stock price will be between 0 and 100 in the first two periods.'}
{'role': 'assistant', 'content': '{\n\t"reasoning": "As it\'s the first time step, I will make a reasonable guess based on the information provided, expecting a moderate start for the stock price.",\n\t"predictedValue1": 50.25,\n\t"predictedValue2": 51.50\n}'}
 
 Time: 0, agent: 1 

{'role': 'user', 'content': 'This is the first time step, give an initial price prediction for the first two periods.  It is very likely that the stock price will be between 0 and 100 in the first two periods.'}
{'role': 'assistant', 'content': '{\n    "reasoning": "I will start with an initial forecast close to the average of the possible range. As there is no historical data available, I will predict 50 for the first two periods.",\n    "predictedValue1": 50,\n    "predicted

In [21]:
def f_bubbles(pe,noise_mean=3,noise_sd=1/4, r=0.05):
    y = np.random.normal(noise_mean,noise_sd)
    R = 1 + r
    pe_mean = np.mean(pe) 
    return (1/R) * pe_mean + y/R

In [22]:
f_bubbles([10, 10, 10], 3, 1)

13.765974768536168

In [ ]:
def f_bubbles(pe,noise_mean=3,noise_sd=1/4, r=0.05):
    y = np.random.normal(noise_mean,noise_sd)
    R = 1 + r
    pe_mean = np.mean(pe) 
    return (1/R) * pe_mean + y/R
    

def f_negative_feedback(pe,noise_mean,noise_sd):
    pe_mean = np.mean(pe)
    epsilon = np.random.normal(noise_mean,noise_sd)
    return 20/21*(123-pe_mean) + epsilon

def f_positive_feedback(pe,noise_mean,noise_sd):
    pe_mean = np.mean(pe)
    epsilon = np.random.normal(noise_mean,noise_sd)
    return 20/21*(pe_mean+3) + epsilon

def earnings(pe,p):
    return np.maximum(1300-1300/49*(pe-p)**2,0)/2600